# Colab 用 llamachat-webui
Run All で起動まで実行します。毎回不要な処理は、関数呼び出しのコメントを外して使います。

このノートだけを Colab に上げて実行できます。残りのファイルは [GitHub](https://github.com/mckey-dev/llamachat-webui) から自動で `git clone` します。

ランタイムを GPU にしてから実行し、起動セルの `*.gradio.live` を開いてください。


## 定数定義


In [ ]:
# 定数定義
import os
from pathlib import Path

os.environ.setdefault("PYTHONUTF8", "1")

# Colab の実体
#   /content/notebooks/          クローン先（セッション限り）
#   /content/tmp/                一時（venv、モデル実体、ログ）
#   Google Drive / llamachat-webui  永続（設定、llama-server、残したいモデル）
if Path("/content").is_dir():
    NOTEBOOK_ROOT = Path("/content/notebooks")
    TEMP_ROOT = Path("/content/tmp")
    SESSION_PERSISTENT = Path("/content/llamachat-webui-runtime")
else:
    NOTEBOOK_ROOT = Path("notebooks")
    TEMP_ROOT = Path("tmp")
    SESSION_PERSISTENT = Path("storage") / "llamachat-webui"

USE_GOOGLE_DRIVE = True
DRIVE_DATA_DIR = "/content/drive/MyDrive/llamachat-webui"

APP_NAME = "llamachat-webui"
APP_REPO = "https://github.com/mckey-dev/llamachat-webui.git"
APP_ROOT = NOTEBOOK_ROOT / APP_NAME

PERSISTENT_ROOT = Path(DRIVE_DATA_DIR) if USE_GOOGLE_DRIVE else SESSION_PERSISTENT
PERSISTENT_APP = PERSISTENT_ROOT
DATA_DIR = PERSISTENT_APP
RUNTIME_DIR = PERSISTENT_APP / "llama-server"

MODELS_REAL = TEMP_ROOT / APP_NAME / "models"
MODELS_PERSISTENT = PERSISTENT_APP / "persistent-models"
MODELS_LINK = DATA_DIR / "models"

TMP_LINK = NOTEBOOK_ROOT / "tmp"
STORAGE_LINK = NOTEBOOK_ROOT / "storage"

VENV_ROOT = TEMP_ROOT / APP_NAME / "venv"
VENV_PYTHON = VENV_ROOT / "bin" / "python"

UI_HOST = "0.0.0.0"
UI_PORT = 7860
UI_SHARE = True
UI_LOG = TEMP_ROOT / APP_NAME / "llamachat.log"
UI_PID = TEMP_ROOT / APP_NAME / "llamachat.pid"

HF_TOKEN = ""
GITHUB_TOKEN = ""

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN.strip()
if GITHUB_TOKEN.strip():
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN.strip()

try:
    from google.colab import userdata
except ImportError:
    userdata = None
if userdata is not None:
    for key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "GITHUB_TOKEN"):
        if os.environ.get(key):
            continue
        try:
            value = userdata.get(key)
        except Exception:
            value = None
        if value:
            os.environ[key] = value
            if key == "HF_TOKEN":
                os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", value)


def show_constants():
    print("【実体】")
    print(f"永続      : {PERSISTENT_APP}")
    print(f"notebooks : {NOTEBOOK_ROOT}")
    print(f"tmp       : {TEMP_ROOT}")
    print(f"Drive     : {'使う' if USE_GOOGLE_DRIVE else '使わない'}")
    print()
    print("【llamachat-webui】")
    print(f"本体          : {APP_ROOT}")
    print(f"データ        : {DATA_DIR}")
    print(f"llama-server  : {RUNTIME_DIR}")
    print(f"models 実体   : {MODELS_REAL}")
    print(f"models 永続   : {MODELS_PERSISTENT}")
    print(f"models リンク : {MODELS_LINK}")
    print(f"リポジトリ    : {APP_REPO}")
    print(f"仮想環境      : {VENV_ROOT}")
    print(f"待ち受け      : {UI_HOST}:{UI_PORT}")
    print("========== 完了 ==========")


show_constants()


## 環境確認
Python、GPU、CUDA、起動に使うパスを確認します。


In [ ]:
# 環境確認
import os
import shutil
import subprocess
import sys
from pathlib import Path


def run_command(command):
    return subprocess.run(command, shell=True, text=True, capture_output=True)


def check_environment():
    for path in (TEMP_ROOT, NOTEBOOK_ROOT):
        path.mkdir(parents=True, exist_ok=True)
    (TEMP_ROOT / APP_NAME).mkdir(parents=True, exist_ok=True)

    print("【実体】")
    print(f"永続      : {PERSISTENT_ROOT}（{'あり' if PERSISTENT_ROOT.exists() else 'なし'}）")
    print(f"notebooks : {NOTEBOOK_ROOT.resolve()}")
    print(f"tmp       : {TEMP_ROOT.resolve()}")

    print()
    print("【Python】")
    print(f"バージョン  : {sys.version.split()[0]}")
    print(f"実行ファイル: {sys.executable}")
    try:
        pip = subprocess.run(
            [sys.executable, "-m", "pip", "--version"],
            capture_output=True,
            text=True,
            check=True,
        )
        print(f"pip         : {pip.stdout.strip()}")
    except Exception as e:
        print(f"pip         : 確認失敗（{e}）")
    if sys.prefix != sys.base_prefix:
        print(f"仮想環境    : 有効（{sys.prefix}）")
    else:
        print("仮想環境    : 未使用（システム Python）")

    print()
    print("【GPU】")
    if shutil.which("nvidia-smi"):
        gpu = run_command(
            "nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader"
        )
        print(gpu.stdout.strip() or "情報を取得できませんでした")
    else:
        print("nvidia-smi が見つかりません")

    print()
    print("【CUDA】")
    cuda_link = Path("/usr/local/cuda")
    print(f"/usr/local/cuda: {cuda_link.resolve() if cuda_link.exists() else 'なし'}")
    print(f"nvcc           : {shutil.which('nvcc') or 'なし'}")
    print(f"CUDA_HOME      : {os.environ.get('CUDA_HOME') or '未設定'}")

    print()
    print("【起動】")
    print(f"本体     : {APP_ROOT}（{'あり' if APP_ROOT.exists() else 'なし'}）")
    print(f"仮想環境 : {VENV_PYTHON}（{'あり' if Path(VENV_PYTHON).exists() else 'なし'}）")
    print(f"models   : {MODELS_REAL}")
    print(f"待ち受け : {UI_HOST}:{UI_PORT}")
    print(f"ログ     : {UI_LOG}")
    print("========== 完了 ==========")


check_environment()


## Google Drive
会話・llama-server 成果物・残したいモデルを Drive に保存します。使わない場合は定数の `USE_GOOGLE_DRIVE` を `False` にしてください。


In [ ]:
# Google Drive
from pathlib import Path


def mount_drive():
    global PERSISTENT_ROOT, PERSISTENT_APP, DATA_DIR, RUNTIME_DIR, MODELS_PERSISTENT, MODELS_LINK

    print("【永続ディレクトリ】")
    if USE_GOOGLE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        PERSISTENT_ROOT = Path(DRIVE_DATA_DIR)
        print("操作   : Google Drive をマウント")
    else:
        PERSISTENT_ROOT = SESSION_PERSISTENT
        print("操作   : セッション限りのディレクトリ")

    PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
    PERSISTENT_APP = PERSISTENT_ROOT
    DATA_DIR = PERSISTENT_APP
    RUNTIME_DIR = PERSISTENT_APP / "llama-server"
    MODELS_PERSISTENT = PERSISTENT_APP / "persistent-models"
    MODELS_LINK = DATA_DIR / "models"
    print(f"永続   : {PERSISTENT_APP}")
    print(f"データ : {DATA_DIR}")
    print("========== 完了 ==========")


mount_drive()


## リポジトリの取得
無い場合は clone します。最新版へ更新する場合は、`update_app()` のコメントを外してください。


In [ ]:
# リポジトリの取得
import subprocess
from pathlib import Path


def git(*args, check=True):
    result = subprocess.run(
        ["git", "-C", str(APP_ROOT), *args],
        capture_output=True,
        text=True,
    )
    if check and result.returncode != 0:
        message = (result.stderr or result.stdout).strip()
        raise RuntimeError(message or f"git {' '.join(args)} に失敗しました")
    return result.stdout.strip()


def git_version(ref="HEAD"):
    result = subprocess.run(
        ["git", "-C", str(APP_ROOT), "describe", "--tags", "--always", ref],
        capture_output=True,
        text=True,
    )
    return result.stdout.strip() if result.returncode == 0 else "不明"


def remote_head_hash():
    result = subprocess.run(
        ["git", "ls-remote", APP_REPO, "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.split()[0]


def fetch_app():
    APP_ROOT.parent.mkdir(parents=True, exist_ok=True)

    print("【llamachat-webui】")
    print(f"保存先 : {APP_ROOT}")

    if APP_ROOT.exists() and not (APP_ROOT / "launch.py").is_file():
        raise RuntimeError(f"launch.py がありません: {APP_ROOT}")

    if not APP_ROOT.exists():
        print("操作   : clone")
        subprocess.run(
            ["git", "clone", "--depth", "1", APP_REPO, str(APP_ROOT)],
            check=True,
        )
    else:
        print("操作   : 既存のためスキップ")

    print(f"バージョン: {git_version() if (APP_ROOT / '.git').exists() else '（git なし）'}")
    print("========== 完了 ==========")


def update_app():
    if not APP_ROOT.exists():
        raise RuntimeError(f"リポジトリがありません: {APP_ROOT}")
    if not (APP_ROOT / ".git").exists():
        raise RuntimeError(f"git リポジトリではありません: {APP_ROOT}")

    print("【更新】")
    local_hash = git("rev-parse", "HEAD")
    print(f"ローカル : {git_version()}")

    try:
        remote_hash = remote_head_hash()
    except Exception as e:
        print(f"リモート : 確認失敗（{e}）")
        print("操作   : スキップ")
        print("========== 完了 ==========")
        return

    if local_hash == remote_hash:
        print("操作   : 最新のためスキップ")
        print("========== 完了 ==========")
        return

    print("操作   : バージョンアップ")
    if MODELS_LINK.is_symlink():
        MODELS_LINK.unlink()
        print("models リンクを一時解除（リンク作成セルを再実行してください）")
    git("fetch", "--depth", "1", "origin")
    remote_ref = git("rev-parse", "--abbrev-ref", "origin/HEAD", check=False)
    if not remote_ref:
        git("remote", "set-head", "origin", "-a")
        remote_ref = git("rev-parse", "--abbrev-ref", "origin/HEAD")
    git("reset", "--hard", remote_ref)
    print(f"更新後 : {git_version()}")
    print("========== 完了 ==========")


fetch_app()
# update_app()


## 仮想環境の作成
一時ストレージに作成します。既にある場合はスキップします。作り直す場合は、`create_venv(recreate=True)` のコメントを外してください。


In [ ]:
# 仮想環境の作成
import os
import shutil
import subprocess
import sys
from pathlib import Path


def venv_python(root):
    for candidate in (root / "bin" / "python", root / "Scripts" / "python.exe"):
        if candidate.exists():
            return candidate
    raise RuntimeError(f"仮想環境の Python が見つかりません: {root}")


def apt_install(packages):
    apt = shutil.which("apt-get")
    if apt is None:
        return
    prefix = []
    if os.geteuid() != 0 and shutil.which("sudo"):
        prefix = ["sudo"]
    subprocess.run([*prefix, "apt-get", "update", "-qq"], check=False)
    subprocess.run([*prefix, "apt-get", "install", "-y", "-qq", *packages], check=False)


def create_venv(recreate=False):
    global VENV_PYTHON

    print("【仮想環境】")
    print(f"作成先 : {VENV_ROOT}")
    print(f"Python : {sys.executable}")

    if VENV_ROOT.exists() and not recreate:
        print("操作   : 既存のためスキップ")
    else:
        if VENV_ROOT.exists():
            print("操作   : 作り直し")
            shutil.rmtree(VENV_ROOT)
        else:
            print("操作   : 作成")
        VENV_ROOT.parent.mkdir(parents=True, exist_ok=True)
        created = subprocess.run([sys.executable, "-m", "venv", str(VENV_ROOT)], check=False)
        if created.returncode != 0 or not (VENV_ROOT / "bin" / "python").exists():
            print("python -m venv に失敗したため python3-venv を入れて再試行します")
            apt_install(["python3-venv"])
            subprocess.run([sys.executable, "-m", "venv", str(VENV_ROOT)], check=True)

    VENV_PYTHON = venv_python(VENV_ROOT)
    version = subprocess.run(
        [str(VENV_PYTHON), "-c", "import sys; print(sys.version.split()[0])"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    print(f"実行ファイル: {VENV_PYTHON}")
    print(f"バージョン  : {version}")
    print("========== 完了 ==========")


create_venv()
# create_venv(recreate=True)


## シンボリックリンクの作成
Drive / tmp を見るリンクと、models の一時実体へのリンクを作ります。


In [ ]:
# シンボリックリンクの作成
import os
import shutil
from pathlib import Path


def copy_dir_structure(src, dst):
    dst.mkdir(parents=True, exist_ok=True)
    for path in src.rglob("*"):
        if path.is_dir() and not path.is_symlink():
            (dst / path.relative_to(src)).mkdir(parents=True, exist_ok=True)


def real_files(path):
    return [p for p in path.rglob("*") if p.is_file() and not p.is_symlink()]


def relative_target(link, real):
    return Path(os.path.relpath(real, start=link.parent)).as_posix()


SKIP_MODEL_DIRS = {".cache", ".ipynb_checkpoints"}


def walk_entries(root):
    found = []
    if not root.exists():
        return found
    stack = [root]
    while stack:
        current = stack.pop()
        try:
            children = list(current.iterdir())
        except OSError:
            continue
        for child in children:
            if child.name in SKIP_MODEL_DIRS:
                continue
            if child.is_symlink():
                found.append(child)
            elif child.is_dir():
                stack.append(child)
            elif child.is_file():
                found.append(child)
    return sorted(found)


def points_to_persistent(path):
    if not path.is_symlink():
        return False
    raw = Path(os.readlink(path))
    target = raw if raw.is_absolute() else path.parent / raw
    try:
        target.resolve().relative_to(MODELS_PERSISTENT.resolve())
    except (OSError, ValueError):
        return False
    return True


def clear_link_path(link, real, *, models=False):
    if link.is_symlink():
        link.unlink()
        print(f"削除   : リンク {link}")
        return

    if not link.exists():
        return

    if not link.is_dir():
        raise RuntimeError(f"ディレクトリではありません: {link}")

    if link.resolve() == real.resolve():
        raise RuntimeError(
            f"リンク位置と実体が同じです: {link}\n"
            "実体は notebooks の外の storage / tmp です。"
        )

    files = real_files(link)
    if models:
        large = [p for p in files if p.stat().st_size > 1024 * 1024]
        if large:
            raise RuntimeError(f"models に実ファイルがあるため削除できません: {large[0]}")
        copy_dir_structure(link, real)
        if files:
            print(f"注意   : プレースホルダ {len(files)} 個は削除されます")
        shutil.rmtree(link)
        return

    if files:
        real.mkdir(parents=True, exist_ok=True)
        for child in list(link.iterdir()):
            dest = real / child.name
            if dest.exists():
                raise RuntimeError(f"実体側に既にあるため移せません: {dest}")
            shutil.move(str(child), str(dest))
        print(f"移動   : {link} の中身 -> {real}")

    shutil.rmtree(link)
    print(f"削除   : 実ディレクトリ {link}")


def create_symlink(link, real, *, models=False):
    real.mkdir(parents=True, exist_ok=True)
    real = real.resolve()
    print(f"リンク : {link}")
    print(f"実体   : {real}")
    clear_link_path(link, real, models=models)
    rel = relative_target(link, real)
    link.symlink_to(rel, target_is_directory=True)
    print(f"結果   : {link} -> {rel}")
    print()


def link_persistent_models():
    MODELS_PERSISTENT.mkdir(parents=True, exist_ok=True)
    MODELS_REAL.mkdir(parents=True, exist_ok=True)

    print("【永続モデル】")
    print(f"永続モデル : {MODELS_PERSISTENT}")

    linked = 0
    skipped = 0
    removed = 0
    wanted = set()

    for src in walk_entries(MODELS_PERSISTENT):
        if src.is_symlink() or not src.is_file():
            continue
        rel = src.relative_to(MODELS_PERSISTENT)
        dest = MODELS_REAL / rel
        wanted.add(rel.as_posix())
        dest.parent.mkdir(parents=True, exist_ok=True)

        if dest.is_symlink():
            same = False
            try:
                same = dest.resolve() == src.resolve()
            except OSError:
                pass
            if same:
                continue
            dest.unlink()
            dest.symlink_to(relative_target(dest, src))
            linked += 1
            continue

        if dest.exists():
            print(f"注意   : 一時側に実ファイルあり {rel.as_posix()}")
            skipped += 1
            continue

        dest.symlink_to(relative_target(dest, src))
        linked += 1

    for path in walk_entries(MODELS_REAL):
        if not path.is_symlink() or not points_to_persistent(path):
            continue
        rel = path.relative_to(MODELS_REAL).as_posix()
        if rel in wanted:
            continue
        path.unlink()
        removed += 1

    print(f"リンク     : {linked} 件")
    print(f"スキップ   : {skipped} 件（一時側に実ファイルあり）")
    print(f"削除       : {removed} 件（元ファイルなし）")
    print()


def unlink_persistent_models():
    print("【永続モデルリンク削除】")
    print(f"対象     : {MODELS_REAL}")
    removed = 0
    for path in walk_entries(MODELS_REAL):
        if not path.is_symlink() or not points_to_persistent(path):
            continue
        path.unlink()
        removed += 1
    print(f"削除     : {removed} 件")
    print("========== 完了 ==========")


def create_symlinks():
    NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    print("【シンボリックリンク】")
    create_symlink(STORAGE_LINK, PERSISTENT_ROOT)
    create_symlink(TMP_LINK, TEMP_ROOT)
    create_symlink(MODELS_LINK, MODELS_REAL, models=True)
    link_persistent_models()
    print("========== 完了 ==========")


create_symlinks()
# unlink_persistent_models()


## パッケージのインストール
venv へ `requirements.txt` を入れます。既にあればスキップします。入れ直す場合は `install_app(force=True)` のコメントを外してください。


In [ ]:
# パッケージのインストール
import subprocess
from pathlib import Path


def venv_can_import(module):
    result = subprocess.run(
        [str(VENV_PYTHON), "-c", f"import {module}"],
        capture_output=True,
        text=True,
        check=False,
    )
    return result.returncode == 0


def install_app(force=False):
    if not Path(VENV_PYTHON).exists():
        raise RuntimeError(f"仮想環境がありません: {VENV_PYTHON}")
    req = APP_ROOT / "requirements.txt"
    if not req.is_file():
        raise RuntimeError(f"requirements.txt がありません: {req}")

    print("【パッケージ】")
    print(f"Python : {VENV_PYTHON}")
    print(f"要件   : {req}")

    if not force and venv_can_import("httpx") and venv_can_import("gradio"):
        print("操作   : 既存のためスキップ")
        print("========== 完了 ==========")
        return

    print("操作   : pip install")
    subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
    result = subprocess.run(
        [str(VENV_PYTHON), "-m", "pip", "install", "-r", str(req)],
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"pip が失敗しました（終了コード {result.returncode}）")
    print("========== 完了 ==========")


install_app()
# install_app(force=True)


## llama-server の準備
既存バイナリがあれば再利用します。無ければ GPU なら CUDA ビルド、失敗時は公式 Vulkan / CPU です。ソースと中間ファイルは捨て、実行ファイルと必要な `.so` だけを永続側へ残します。作り直す場合は `ensure_llama_server(force=True)` のコメントを外してください。


In [ ]:
# llama-server の準備
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

LLAMA_CPP_REPO = "https://github.com/ggml-org/llama.cpp.git"
RUNTIME_LIB_PREFIXES = ("libggml", "libllama", "libmtmd")


def run(cmd, *, cwd=None, check=True, env=None):
    printable = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"+ {printable}", flush=True)
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        check=check,
        text=True,
        shell=isinstance(cmd, str),
        env=env,
    )


def apt_install(packages):
    apt = shutil.which("apt-get")
    if apt is None:
        return
    prefix = []
    if os.geteuid() != 0 and shutil.which("sudo"):
        prefix = ["sudo"]
    run([*prefix, "apt-get", "update", "-qq"], check=False)
    run([*prefix, "apt-get", "install", "-y", "-qq", *packages], check=False)


def gpu_available():
    if shutil.which("nvidia-smi") is None:
        return False
    result = subprocess.run(
        ["nvidia-smi", "-L"],
        capture_output=True,
        text=True,
        timeout=15,
        check=False,
    )
    return result.returncode == 0 and bool(result.stdout.strip())


def find_nvcc():
    found = shutil.which("nvcc")
    if found:
        return found
    fallback = Path("/usr/local/cuda/bin/nvcc")
    return str(fallback) if fallback.is_file() else None


def is_runtime_artifact(path):
    name = path.name
    if name in {"llama-server", "BUILD.txt"}:
        return True
    if name.startswith(RUNTIME_LIB_PREFIXES):
        return True
    if path.suffix == ".so" or ".so." in name:
        return True
    return False


def is_under(path, root):
    try:
        path.resolve().relative_to(root.resolve())
        return True
    except ValueError:
        return False


def collect_runtime_files(binary, *, build_root=None):
    found = {binary.name: binary}
    search_dirs = [binary.parent]
    if build_root:
        search_dirs.extend([build_root / "build" / "bin", build_root / "bin"])
    for directory in search_dirs:
        if not directory.is_dir():
            continue
        for item in directory.iterdir():
            if item.is_file() and item.name.startswith(RUNTIME_LIB_PREFIXES):
                found[item.name] = item
    ldd = shutil.which("ldd")
    if ldd:
        result = subprocess.run([ldd, str(binary)], capture_output=True, text=True, check=False)
        for line in result.stdout.splitlines():
            if "=>" not in line:
                continue
            raw = line.split("=>", 1)[1].strip().split()
            if not raw or raw[0] == "not":
                continue
            lib = Path(raw[0])
            if not lib.is_file():
                continue
            if lib.name.startswith(RUNTIME_LIB_PREFIXES) or (build_root and is_under(lib, build_root)):
                found[lib.name] = lib
    return list(found.values())


def persist_llama_runtime(binary, dest_dir, *, build_root=None, note=""):
    dest_dir.mkdir(parents=True, exist_ok=True)
    for src in collect_runtime_files(binary, build_root=build_root):
        target = dest_dir / src.name
        if src.resolve() != target.resolve():
            shutil.copy2(src, target)
        try:
            target.chmod(target.stat().st_mode | 0o111)
        except OSError:
            pass
    saved = dest_dir / "llama-server"
    if note:
        (dest_dir / "BUILD.txt").write_text(note.rstrip() + "\n", encoding="utf-8")
    print("残したファイル:")
    for child in sorted(dest_dir.iterdir()):
        if is_runtime_artifact(child):
            print(f"  {child.name}")
    if not saved.is_file():
        raise RuntimeError("llama-server を永続ディレクトリへコピーできませんでした。")
    return saved


def usable_llama_server(path):
    if not path.is_file() or path.name not in {"llama-server", "llama-server.exe"}:
        return None
    try:
        path.chmod(path.stat().st_mode | 0o111)
    except OSError:
        pass
    return path


def find_existing_runtime():
    candidates = [
        RUNTIME_DIR / "llama-server",
        RUNTIME_DIR,
        DATA_DIR / "llama-server" / "llama-server",
        PERSISTENT_APP / "llama-server" / "llama-server",
    ]
    settings = DATA_DIR / "settings.json"
    if settings.is_file():
        try:
            raw = json.loads(settings.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            raw = {}
        saved = str((raw or {}).get("llama_server_path") or "").strip()
        if saved:
            candidates.append(Path(saved))
    if RUNTIME_DIR.is_dir():
        candidates.extend(p for p in RUNTIME_DIR.rglob("llama-server") if p.is_file())

    seen = set()
    for path in candidates:
        try:
            resolved = path.expanduser().resolve()
        except OSError:
            continue
        key = str(resolved)
        if key in seen:
            continue
        seen.add(key)
        usable = usable_llama_server(resolved)
        if usable:
            return usable
    return None


def prune_persistent_junk():
    for name in ("src", ".download", "llama.cpp"):
        path = PERSISTENT_APP / name
        if path.exists():
            print(f"削除 : {path}")
            shutil.rmtree(path, ignore_errors=True)
    if not RUNTIME_DIR.is_dir():
        return
    for child in list(RUNTIME_DIR.iterdir()):
        if is_runtime_artifact(child):
            continue
        print(f"削除 : {child}")
        if child.is_dir():
            shutil.rmtree(child, ignore_errors=True)
        else:
            child.unlink(missing_ok=True)


def scratch_dir(name):
    path = TEMP_ROOT / APP_NAME / name
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    print(f"作業ディレクトリ: {path}", flush=True)
    return path


def reuse_llama_server():
    existing = find_existing_runtime()
    if existing is None:
        print("再利用できる llama-server はありません")
        return None
    dest = RUNTIME_DIR
    if existing.parent.resolve() != dest.resolve() and existing.resolve() != dest.resolve():
        print(f"発見 : {existing} → {dest} へコピー")
        existing = persist_llama_runtime(
            existing,
            dest,
            build_root=existing.parent,
            note="backend=cuda\nmigrated=1\n",
        )
    print(f"再利用 : {existing}")
    return existing


def build_cuda_llama_server():
    nvcc = find_nvcc()
    if not nvcc:
        raise RuntimeError("nvcc がありません。CUDA toolkit 付きの GPU ランタイムを選んでください。")
    print(f"CUDA ビルド先: {RUNTIME_DIR}", flush=True)
    apt_install(["build-essential", "cmake", "git", "pciutils"])
    tmp = scratch_dir("llama.cpp-src")
    src_dir = tmp / "llama.cpp"
    try:
        run(["git", "clone", "--depth", "1", LLAMA_CPP_REPO, str(src_dir)])
        env = os.environ.copy()
        env["PATH"] = str(Path(nvcc).parent) + os.pathsep + env.get("PATH", "")
        jobs = str(max(1, min(os.cpu_count() or 2, 8)))
        cmake = shutil.which("cmake") or "cmake"
        run(
            [
                cmake,
                "-B",
                "build",
                "-DGGML_CUDA=ON",
                "-DCMAKE_BUILD_TYPE=Release",
                "-DCMAKE_CUDA_ARCHITECTURES=75;80;86;89;90",
            ],
            cwd=src_dir,
            env=env,
        )
        run(
            [cmake, "--build", "build", "--config", "Release", "-j", jobs, "--target", "llama-server"],
            cwd=src_dir,
            env=env,
        )
        built = src_dir / "build" / "bin" / "llama-server"
        if not built.is_file():
            hits = [p for p in (src_dir / "build").rglob("llama-server") if p.is_file()]
            if not hits:
                raise RuntimeError("ビルド後に llama-server が見つかりません。")
            built = hits[0]
        rev = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"],
            cwd=str(src_dir),
            check=False,
            capture_output=True,
            text=True,
        ).stdout.strip()
        note = f"backend=cuda\nsource={LLAMA_CPP_REPO}\nrevision={rev or 'unknown'}\n"
        return persist_llama_runtime(built, RUNTIME_DIR, build_root=src_dir, note=note)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def install_official_backend(backend):
    if str(APP_ROOT) not in sys.path:
        sys.path.insert(0, str(APP_ROOT))
    from app.install_server import install_blocking

    tmp = scratch_dir("llama-official")
    try:
        binary = Path(install_blocking(tmp, backend=backend))
        return persist_llama_runtime(
            binary,
            RUNTIME_DIR,
            build_root=tmp,
            note=f"backend={backend}\nsource=github-release\n",
        )
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def ensure_llama_server(force=False):
    print("【llama-server】")
    print(f"永続先 : {RUNTIME_DIR}")
    binary = None

    if not force:
        binary = reuse_llama_server()

    if binary is None and gpu_available():
        try:
            binary = build_cuda_llama_server()
        except Exception as exc:
            print(f"CUDA ビルド失敗（{exc}）。公式 Vulkan / CPU に切り替えます。")

    if binary is None:
        backend = "vulkan" if gpu_available() else "cpu"
        try:
            binary = install_official_backend(backend)
        except Exception as exc:
            if backend != "cpu":
                print(f"{backend} 導入失敗（{exc}）。CPU に切り替えます。")
                binary = install_official_backend("cpu")
            else:
                raise

    prune_persistent_junk()
    os.environ["LLAMA_SERVER"] = str(binary)
    print(f"llama-server : {binary}")
    print("========== 完了 ==========")
    return binary


ensure_llama_server()
# ensure_llama_server(force=True)


## UI の起動
Gradio の公開 URL（`*.gradio.live`）で開きます。このセルは起動確認が終わるまで待ち、その後は完了します。


In [ ]:
# UI の起動
import os
import re
import socket
import subprocess
import time
import urllib.request
from pathlib import Path

from IPython.display import Markdown, display


def port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0


def http_ok():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{UI_PORT}/", timeout=2) as resp:
            return 200 <= resp.status < 500
    except Exception:
        return False


def log_tail(path, n=80):
    if not path.exists():
        return "(ログなし)"
    text = path.read_text(encoding="utf-8", errors="replace").strip()
    if not text:
        return "(空)"
    return "\n".join(text.splitlines()[-n:])


def show_log():
    print("【ログ】")
    print(log_tail(UI_LOG))


def share_urls():
    if not UI_LOG.exists():
        return []
    text = UI_LOG.read_text(encoding="utf-8", errors="replace")
    found = re.findall(r"https://[a-zA-Z0-9.-]+\.gradio\.live", text)
    return list(dict.fromkeys(found))

def show_open_url():
    urls = share_urls()
    if urls:
        for share in urls:
            print(f"URL      : {share}")
            display(Markdown(f"[llamachat-webui を開く]({share})"))
        return
    print("開き方   : ログの https://….gradio.live を開く")

def ui_cmd():
    cmd = [
        str(VENV_PYTHON),
        str(APP_ROOT / "launch.py"),
        "--server-name",
        UI_HOST,
        "--server-port",
        str(UI_PORT),
        "--data-dir",
        str(DATA_DIR),
        "--skip-install",
    ]
    if UI_SHARE:
        cmd.append("--share")
    return cmd


def spawn(cmd):
    UI_LOG.parent.mkdir(parents=True, exist_ok=True)
    log_f = open(UI_LOG, "wb", buffering=0)
    env = os.environ.copy()
    env.pop("PYTHONSAFEPATH", None)
    previous = env.get("PYTHONPATH", "").strip()
    env["PYTHONPATH"] = str(APP_ROOT) if not previous else str(APP_ROOT) + os.pathsep + previous
    proc = subprocess.Popen(
        cmd,
        cwd=str(APP_ROOT),
        stdout=log_f,
        stderr=subprocess.STDOUT,
        start_new_session=True,
        env=env,
    )
    UI_PID.write_text(str(proc.pid), encoding="utf-8")
    return proc, log_f


def wait_ready(proc, log_f, timeout=180):
    deadline = time.time() + timeout
    http_at = None
    while time.time() < deadline:
        if proc.poll() is not None:
            log_f.close()
            return False
        if http_ok():
            if http_at is None:
                http_at = time.time()
            share_wait = 60 if UI_SHARE else 0
            if not UI_SHARE or share_urls() or time.time() - http_at >= share_wait:
                return True
        time.sleep(1)
    return http_at is not None


def start_ui():
    if not Path(VENV_PYTHON).exists():
        raise RuntimeError(f"仮想環境がありません: {VENV_PYTHON}")
    if not (APP_ROOT / "launch.py").is_file():
        raise RuntimeError(f"launch.py がありません: {APP_ROOT}")

    print("【起動】")
    print(f"待ち受け : {UI_HOST}:{UI_PORT}")
    print(f"データ   : {DATA_DIR}")

    if http_ok():
        print("操作     : 既に起動中のため再利用")
        show_open_url()
        print("========== 完了 ==========")
        return

    if port_open(UI_PORT):
        raise RuntimeError(f"ポート {UI_PORT} は使用中です")

    proc, log_f = spawn(ui_cmd())
    print(f"操作     : 起動（pid {proc.pid}）")

    if wait_ready(proc, log_f):
        show_open_url()
        print("========== 完了 ==========")
        return

    show_log()
    raise RuntimeError(f"起動に失敗しました。ログ: {UI_LOG}")


start_ui()


## UI の停止
必要なときだけ、`stop_ui()` のコメントを外してください。Run All では呼びません。


In [ ]:
# UI の停止
import os
import signal
import socket
import subprocess
import time
from pathlib import Path


def read_pid(path):
    if not path.exists():
        return None
    try:
        return int(path.read_text(encoding="utf-8").strip())
    except ValueError:
        return None


def pid_alive(pid):
    if not pid:
        return False
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False


def port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0


def kill_pid(pid):
    try:
        os.killpg(pid, signal.SIGTERM)
    except OSError:
        os.kill(pid, signal.SIGTERM)
    for _ in range(20):
        if not pid_alive(pid):
            return "停止"
        time.sleep(0.25)
    try:
        os.killpg(pid, signal.SIGKILL)
    except OSError:
        os.kill(pid, signal.SIGKILL)
    return "強制停止"


def stop_ui():
    print("【停止】")
    pid = read_pid(Path(UI_PID))
    if pid and pid_alive(pid):
        result = kill_pid(pid)
        print(f"操作 : {result}（pid {pid}）")
    elif pid:
        print(f"操作 : 既に停止（pid {pid}）")
    else:
        print("操作 : PID なし")

    if Path(UI_PID).exists():
        Path(UI_PID).unlink()

    if port_open(UI_PORT):
        subprocess.run(
            ["fuser", "-k", f"{UI_PORT}/tcp"],
            capture_output=True,
            text=True,
        )
        time.sleep(0.5)

    print(f"ポート : {'使用中' if port_open(UI_PORT) else '空き'}（{UI_PORT}）")
    print("========== 完了 ==========")


# stop_ui()


## 掃除
`.ipynb_checkpoints`、`.cache`、Hugging Face キャッシュを削除します。
必要なときだけ、`cleanup_junk()` のコメントを外してください。Run All では呼びません。


In [ ]:
# 掃除
import os
import shutil


def remove_dirs(paths):
    count = 0
    for path in paths:
        if path.is_dir():
            shutil.rmtree(path, ignore_errors=True)
            count += 1
    return count


def cleanup_junk():
    print("【掃除】")
    cache_dirs = []
    if NOTEBOOK_ROOT.exists():
        cache_dirs.extend(p for p in NOTEBOOK_ROOT.rglob(".ipynb_checkpoints") if p.is_dir())
        cache_dirs.extend(p for p in NOTEBOOK_ROOT.rglob(".cache") if p.is_dir())
    print(f"notebooks キャッシュ : {remove_dirs(cache_dirs)} 件")

    for path in (MODELS_REAL / ".cache", TEMP_ROOT / APP_NAME / "llama.cpp-src"):
        shutil.rmtree(path, ignore_errors=True)
        print(f"削除 : {path}")

    cache_dir = os.getenv("HUGGINGFACE_HUB_CACHE")
    if cache_dir is None:
        cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
    shutil.rmtree(cache_dir, ignore_errors=True)
    print(f"HF cache : {cache_dir}")
    print("========== 完了 ==========")


# cleanup_junk()


## 予備
必要なときだけ使います。Run All では何もしません。


In [ ]:
# 予備
